<a href="https://colab.research.google.com/github/iamsakshi3006/Expense-tracker/blob/main/Smart_City_Traffic_Flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

In [3]:
SEED = 42
rng = np.random.default_rng(SEED)

DAYS = 30
JUNCTIONS = 10
HOURS = 24

In [4]:
baseline_flow = np.array([35, 42, 50, 55, 38, 62, 45, 70, 48, 58], dtype=float)

hour_labels = np.array([f"Junction {i:02d}" for i in range(1, JUNCTIONS + 1)])

print("Days:", DAYS)
print("Junctions:", JUNCTIONS)
print("Hours per day:", HOURS)
print("Total junction-hour records:", DAYS * JUNCTIONS * HOURS)

Days: 30
Junctions: 10
Hours per day: 24
Total junction-hour records: 7200


In [5]:
rush_curve = np.array([
    0.25, 0.20, 0.18, 0.18, 0.22, 0.35, 0.40 ,1.25, 1.70, 1.40, 1.05, 0.95, 0.95, 0.95, 0.98, 1.05, 1.30, 1.70, 1.90, 1.65, 1.25, 0.85, 0.55, 0.35

], dtype=float)

print("Hour | Traffic multiplier")
print("-----|--------------------")
for hour, multiplier in enumerate(rush_curve):
  print(f"{hour:02d}:00 | {multiplier:.2f}")

  print("\nMorning rush window: 06:00 to 10:00")
  print("Evening rush window: 16:00 to 20:00")

Hour | Traffic multiplier
-----|--------------------
00:00 | 0.25

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
01:00 | 0.20

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
02:00 | 0.18

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
03:00 | 0.18

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
04:00 | 0.22

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
05:00 | 0.35

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
06:00 | 0.40

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
07:00 | 1.25

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
08:00 | 1.70

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
09:00 | 1.40

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
10:00 | 1.05

Morning rush window: 06:00 to 10:00
Evening rush window: 16:00 to 20:00
1

In [6]:
expected_rate = baseline_flow[:, np.newaxis] * rush_curve[np.newaxis, :]

traffic = rng.poisson(
    lam = expected_rate[np.newaxis, :, :],
    size = (DAYS, JUNCTIONS, HOURS)

)

print("Traffic array shape:", traffic.shape)
print("Minimum hourly vehicle count:", traffic.min())
print("Maximum hourly vehicle count:", traffic.max())
print("Total simulated vehicles:", traffic.sum())

assert traffic.shape == (30, 10, 24)
assert traffic.size == 7200

Traffic array shape: (30, 10, 24)
Minimum hourly vehicle count: 0
Maximum hourly vehicle count: 157
Total simulated vehicles: 325966


In [7]:
record_dtype = np.dtype([
    ("day", np.int16),
    ("junction", np.int8),
    ("hour", np.int8),
    ("vehicle_count", np.int32)
])

records = np.empty(DAYS * JUNCTIONS * HOURS, dtype=record_dtype)
records["day"] = np.repeat(np.arange(1, DAYS + 1), JUNCTIONS * HOURS)
records["junction"] = np.tile(
    np.repeat(np.arange(1, JUNCTIONS + 1), HOURS), DAYS
)
records["hour"] = np.tile(np.arange(HOURS), DAYS * JUNCTIONS)
records["vehicle_count"] = traffic.reshape(-1)

print("Dataset records:", records.size)
print("Dataset fields:", records.dtype.names)
print("\nFirst 10 records:")
print(records[:10])

counts = records["vehicle_count"]
print("\nVehicle-count summary")
print("Mean:", round(float(np.mean(counts)), 2))
print("Median:", float(np.median(counts)))
print("Minimum:", int(np.min(counts)))
print("Maximum:", int(np.max(counts)))
print("Standard deviation:", round(float(np.std(counts)), 2))

assert records.size == 7200


Dataset records: 7200
Dataset fields: ('day', 'junction', 'hour', 'vehicle_count')

First 10 records:
[(1, 1, 0, 12) (1, 1, 1,  8) (1, 1, 2,  6) (1, 1, 3,  7) (1, 1, 4, 10)
 (1, 1, 5, 14) (1, 1, 6, 10) (1, 1, 7, 50) (1, 1, 8, 64) (1, 1, 9, 48)]

Vehicle-count summary
Mean: 45.27
Median: 44.0
Minimum: 0
Maximum: 157
Standard deviation: 29.93


In [8]:
daily_hourly_totals = traffic.sum(axis=1)

average_hourly_total = daily_hourly_totals.mean(axis=0)

overall_peak_hour = int(np.argmax(average_hourly_total))
morning_hours = np.arange(6, 11)
evening_hours = np.arange(16, 21)

morning_peak_hour = int(
    morning_hours[np.argmax(average_hourly_total[morning_hours])]
)
evening_peak_hour = int(
    evening_hours[np.argmax(average_hourly_total[evening_hours])]
)

def format_hour(hour):
    return f"{hour:02d}:00"

print("Hour | Average vehicles across all junctions")
print("-----|-------------------------------------")
for hour, total in enumerate(average_hourly_total):
    print(f"{format_hour(hour)} | {total:8.2f}")

    print("\nOverall busiest hour:", format_hour(overall_peak_hour))
print("Morning rush peak:", format_hour(morning_peak_hour))
print("Evening rush peak:", format_hour(evening_peak_hour))


Hour | Average vehicles across all junctions
-----|-------------------------------------
00:00 |   125.73

Overall busiest hour: 18:00
01:00 |    99.47

Overall busiest hour: 18:00
02:00 |    89.87

Overall busiest hour: 18:00
03:00 |    92.33

Overall busiest hour: 18:00
04:00 |   108.27

Overall busiest hour: 18:00
05:00 |   173.43

Overall busiest hour: 18:00
06:00 |   205.57

Overall busiest hour: 18:00
07:00 |   633.77

Overall busiest hour: 18:00
08:00 |   865.00

Overall busiest hour: 18:00
09:00 |   703.13

Overall busiest hour: 18:00
10:00 |   519.30

Overall busiest hour: 18:00
11:00 |   478.07

Overall busiest hour: 18:00
12:00 |   471.40

Overall busiest hour: 18:00
13:00 |   480.33

Overall busiest hour: 18:00
14:00 |   496.67

Overall busiest hour: 18:00
15:00 |   523.07

Overall busiest hour: 18:00
16:00 |   652.30

Overall busiest hour: 18:00
17:00 |   852.27

Overall busiest hour: 18:00
18:00 |   958.67

Overall busiest hour: 18:00
19:00 |   825.50

Overall busiest hou

In [13]:
capacity_per_junction = baseline_flow * 1.6

over_capacity = traffic > capacity_per_junction[np.newaxis, :, np.newaxis]
over_capacity_pct = over_capacity.mean(axis=(0, 2)) * 100
average_vehicles_per_hour = traffic.mean(axis=(0, 2))
peak_hourly_average = traffic.mean(axis=0).max(axis=1)
total_vehicles_30_days = traffic.sum(axis=(0, 2))
average_vehicles_per_day = total_vehicles_30_days / DAYS

order = np.lexsort((-average_vehicles_per_day, -over_capacity_pct))

print("POTENTIAL BOTTLENECKS — RANKED JUNCTIONS")
print("Rank | Junction   | Avg/hour | Avg/day | Peak hour avg | Capacity | Over-capacity %")
print("-----|------------|----------|---------|---------------|----------|----------------")
for rank, idx in enumerate(order, start=1):
    print(
        f"{rank:>4} | {hour_labels[idx]:10} | "
        f"{average_vehicles_per_hour[idx]:8.2f} | "
        f"{average_vehicles_per_day[idx]:7.2f} | "
        f"{peak_hourly_average[idx]:13.2f} | "
        f"{capacity_per_junction[idx]:8.2f} | "
        f"{over_capacity_pct[idx]:15.2f}%"
    )

busiest_junction_index = int(order[0])
print("\nHighest-ranked potential bottleneck:", hour_labels[busiest_junction_index])


POTENTIAL BOTTLENECKS — RANKED JUNCTIONS
Rank | Junction   | Avg/hour | Avg/day | Peak hour avg | Capacity | Over-capacity %
-----|------------|----------|---------|---------------|----------|----------------
   1 | Junction 02 |    37.98 |  911.63 |         78.53 |    67.20 |           13.89%
   2 | Junction 05 |    34.63 |  831.07 |         71.00 |    60.80 |           13.75%
   3 | Junction 10 |    52.30 | 1255.23 |        113.70 |    92.80 |           13.47%
   4 | Junction 07 |    40.54 |  972.93 |         86.17 |    72.00 |           13.19%
   5 | Junction 04 |    49.40 | 1185.60 |        103.43 |    88.00 |           13.06%
   6 | Junction 03 |    45.29 | 1086.87 |         98.43 |    80.00 |           12.92%
   7 | Junction 09 |    43.03 | 1032.77 |         90.47 |    76.80 |           12.78%
   8 | Junction 01 |    31.46 |  755.07 |         65.33 |    56.00 |           12.22%
   9 | Junction 06 |    55.55 | 1333.30 |        116.73 |    99.20 |           11.94%
  10 | Junction 0

In [14]:
junction_hour_average = traffic.mean(axis=0)

heat_symbols = np.array(list(" .:-=+*#%@"))
minimum = junction_hour_average.min()
maximum = junction_hour_average.max()

if maximum == minimum:
    intensity_indices = np.zeros(junction_hour_average.shape, dtype=int)
else:
    scaled = (junction_hour_average - minimum) / (maximum - minimum)
    intensity_indices = np.rint(scaled * (len(heat_symbols) - 1)).astype(int)
    intensity_indices = np.clip(intensity_indices, 0, len(heat_symbols) - 1)

print("ASCII TRAFFIC HEATMAP — rows: junctions, columns: hours")
print("Low", repr(heat_symbols[0]), "  ->  High", repr(heat_symbols[-1]))
print("Junction  |", " ".join(f"{hour:02d}" for hour in range(HOURS)))
print("----------+" + "---" * HOURS)

for junction_index in range(JUNCTIONS):
    row_symbols = "".join(heat_symbols[int(value)] for value in intensity_indices[junction_index])
    print(f"{hour_labels[junction_index]:10}| {row_symbols}")

print("\nAverage count range: "
      f"{minimum:.2f} to {maximum:.2f} vehicles per hour")


ASCII TRAFFIC HEATMAP — rows: junctions, columns: hours
Low np.str_(' ')   ->  High np.str_('@')
Junction  | 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
----------+------------------------------------------------------------------------
Junction 01|      ..-=-::::::-===-:. 
Junction 02|      ..-+=:-:::-=+++-:..
Junction 03| .    ..=*+------=+*+=-:.
Junction 04| .    ..+*+=----=+*#*=-:.
Junction 05|      ..-=-::::::-=+=-:..
Junction 06| .   ...+#*======+#%#+-:.
Junction 07|      ..-+=------=+*+=:..
Junction 08| ......:*%*+====+*%@%*=:.
Junction 09| .    ..=+=------=+*+=::.
Junction 10| ..  ...+#+==--==+#%*+-:.

Average count range: 4.97 to 134.87 vehicles per hour


In [15]:
def simulate_loop(seed=123):
    local_rng = np.random.default_rng(seed)
    result = np.empty((DAYS, JUNCTIONS, HOURS), dtype=np.int32)
    for day in range(DAYS):
        for junction in range(JUNCTIONS):
            for hour in range(HOURS):
                result[day, junction, hour] = local_rng.poisson(
                    expected_rate[junction, hour]
                )
    return result


def simulate_vectorized(seed=123):
    local_rng = np.random.default_rng(seed)
    return local_rng.poisson(
        lam=expected_rate[np.newaxis, :, :],
        size=(DAYS, JUNCTIONS, HOURS)
    )

loop_result = simulate_loop()
vectorized_result = simulate_vectorized()

print("Loop result shape:", loop_result.shape)
print("Vectorized result shape:", vectorized_result.shape)
assert loop_result.shape == vectorized_result.shape == (DAYS, JUNCTIONS, HOURS)

print("\nLoop-based timing:")
%timeit -n 3 -r 5 simulate_loop()

print("\nVectorized NumPy timing:")
%timeit -n 3 -r 5 simulate_vectorized()


Loop result shape: (30, 10, 24)
Vectorized result shape: (30, 10, 24)

Loop-based timing:
14.8 ms ± 542 µs per loop (mean ± std. dev. of 5 runs, 3 loops each)

Vectorized NumPy timing:
1.08 ms ± 82.8 µs per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [16]:
print("SMART CITY TRAFFIC FLOW SIMULATOR — FINAL SUMMARY")
print("-" * 58)
print(f"Dataset records: {records.size:,} junction-hour records")
print(f"Simulation period: {DAYS} days")
print(f"Number of junctions: {JUNCTIONS}")
print(f"Hours per day: {HOURS}")
print(f"Total simulated vehicles: {int(traffic.sum()):,}")
print(f"Overall busiest hour: {format_hour(overall_peak_hour)}")
print(f"Morning rush peak: {format_hour(morning_peak_hour)}")
print(f"Evening rush peak: {format_hour(evening_peak_hour)}")
print(
    "Highest-ranked potential bottleneck: "
    f"{hour_labels [busiest_junction_index]} "
    f"({over_capacity_pct[busiest_junction_index]:.2f}% of simulated hours "
    "above its assumed capacity)"
)
print("\nNote: Results are based on synthetic Poisson traffic data.")
print("Capacity values are illustrative assumptions, not real measurements.")


SMART CITY TRAFFIC FLOW SIMULATOR — FINAL SUMMARY
----------------------------------------------------------
Dataset records: 7,200 junction-hour records
Simulation period: 30 days
Number of junctions: 10
Hours per day: 24
Total simulated vehicles: 325,966
Overall busiest hour: 18:00
Morning rush peak: 08:00
Evening rush peak: 18:00
Highest-ranked potential bottleneck: Junction 02 (13.89% of simulated hours above its assumed capacity)

Note: Results are based on synthetic Poisson traffic data.
Capacity values are illustrative assumptions, not real measurements.
